# Week 98, Day 4: Debugging interview

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w098d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
multipliers = [lambda x: x * i for i in range(3)]
print([m(10) for m in multipliers])

> **Key idea.** Read the error (last line, innermost frame), reproduce small, hypothesise aloud, test with one experiment, fix the cause, add a test. Watch for late-binding closures (bind with a default argument), changing a list while iterating (build a new one), float equality (`isclose`, `Decimal`), numbers read as text (convert first), and shadowed built-ins.

### Exercise: Reading a traceback

Write `innermost(traceback_text)`: from a standard traceback, return a dict with `file`, `line` (an int) and `function` from the **last** `File "...", line N, in NAME` line, and `error` and `message` from the last line (split at the first `": "`; the message is `""` if there's none).

In [ ]:
import re

def innermost(traceback_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TB = \"\"\"Traceback (most recent call last):\n  File \"/home/ada/waymark/cli.py\", line 41, in main\n    report = monthly(walks)\n  File \"/home/ada/waymark/stats.py\", line 17, in monthly\n    totals[month] += walk[\"km\"]\nKeyError: '2026-02'\"\"\"\n\ndef test_innermost():\n    \"where it really happened\"\n    assert innermost(TB) == {\"file\": \"/home/ada/waymark/stats.py\", \"line\": 17, \"function\": \"monthly\",\n                             \"error\": \"KeyError\", \"message\": \"'2026-02'\"}\n    tb2 = 'Traceback (most recent call last):\\n  File \"<stdin>\", line 1, in <module>\\nStopIteration'\n    assert innermost(tb2) == {\"file\": \"<stdin>\", \"line\": 1, \"function\": \"<module>\", \"error\": \"StopIteration\", \"message\": \"\"}")

### Exercise: Narrowing the input

A function fails on a long list of inputs, and once it fails on a prefix it fails on every longer prefix. Write `first_failing_prefix(items, fails)`: the smallest `n` such that `fails(items[:n])` is true, found by binary search, and the number of calls made, as `(n, calls)`; `(None, calls)` if even the whole list doesn't fail.

In [ ]:
def first_failing_prefix(items, fails):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_narrow():\n    \"the walk that broke it\"\n    walks = [{\"km\": 5}] * 700 + [{\"km\": None}] + [{\"km\": 3}] * 299\n    fails = lambda part: any(w[\"km\"] is None for w in part)\n    n, calls = first_failing_prefix(walks, fails)\n    assert n == 701 and calls <= 11, (n, calls)\n    assert first_failing_prefix([1, 2, 3], lambda p: False)[0] is None\n    assert first_failing_prefix([0], lambda p: 0 in p) == (1, 1)")

### Exercise: Functions made in a loop

`make_multipliers(n)` should return `n` functions where the `i`-th multiplies by `i`, but every one multiplies by `n - 1`. Fix it.

In [ ]:
def make_multipliers(n):
    return [lambda x: x * i for i in range(n)]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_closures():\n    \"bound now, not later\"\n    assert [m(10) for m in make_multipliers(3)] == [0, 10, 20] and make_multipliers(0) == []")

### Exercise: Removing while looping

`remove_short(words, n)` should remove every word shorter than `n` and return the list, but removing while looping skips the word after each removal. Fix it so it returns a new list and leaves `words` unchanged.

In [ ]:
def remove_short(words, n):
    for w in words:
        if len(w) < n:
            words.remove(w)
    return words

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_remove():\n    \"nothing skipped\"\n    words = [\"a\", \"be\", \"walk\", \"up\", \"hill\", \"to\"]\n    assert remove_short(words, 3) == [\"walk\", \"hill\"] and words == [\"a\", \"be\", \"walk\", \"up\", \"hill\", \"to\"]")

### Exercise: Totals that didn't add up

`matches_total(prices, total)` compares a sum of floats with `==`, so `[0.1, 0.2]` doesn't match `0.3`. Fix it to compare to within half a penny (`abs_tol=0.005`) with `math.isclose`.

In [ ]:
import math

def matches_total(prices, total):
    return sum(prices) == total

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_floats():\n    \"close enough, and no closer\"\n    assert matches_total([0.1, 0.2], 0.3) and matches_total([19.99, 5.01], 25.0)\n    assert not matches_total([10.0, 2.0], 12.01) and matches_total([], 0)")

### Bug hunt: The oldest walker is nine?

The ages came from a form as text, and as text `"9"` is greater than `"10"`, so this prints `9`. Compare them as numbers so it prints `72`.

In [ ]:
ages = ["34", "9", "72", "10"]
print(max(ages))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"convert first\"\n    expect_output(\"72\")")

### Bug hunt: list is not callable

The name `list` was reused for a variable, so the later call `list(...)` fails with `TypeError: 'list' object is not callable`. Rename the variable so it prints `['Mill', 'Weir']`.

In [ ]:
list = ["Mill", "Weir", "Mill"]
unique = list(dict.fromkeys(list))
print(unique)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"don't shadow built-ins\"\n    expect_output(\"['Mill', 'Weir']\")")

### Puzzle: Narrowing by halves

Arrange the loop of `first_failing_prefix`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # [m(1) for m in [lambda x: x + i for i in range(2)]]
q = ___   # 0.1 + 0.2 == 0.3
r = ___   # "9" > "10"
s = ___   # the frame of a traceback where the error happened: "first" or "last"
t = ___   # the built-in function that starts the debugger

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [2, 2] and q is False and r is True, \"p, q and r\"\n    assert s == \"last\" and t in (\"breakpoint\", \"breakpoint()\"), \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny post-mortem debugger

Write `post_mortem(source, call)`: compile `source` with the file name `"<debugged>"`, execute it into a namespace, then evaluate the expression `call` in that namespace. If it raises, return lines: `"ERROR: MESSAGE"` (the exception's class name and its `str`), `"at line N in FUNCTION: TEXT"` for the **innermost** frame from `"<debugged>"` (TEXT the stripped source line), and `"  NAME = VALUE"` for each local variable of that frame, sorted by name, VALUE written with `repr`. If it doesn't raise, return `["no error: RESULT"]` with the result's `repr`.

In [ ]:
import sys
import traceback

def post_mortem(source, call):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SOURCE = '''def monthly(walks):\n    totals = {}\n    for walk in walks:\n        month = walk[\"day\"][:7]\n        totals[month] += walk[\"km\"]\n    return totals\n\ndef report(walks):\n    return monthly(walks)\n'''\n\ndef test_error():\n    \"where, and with what\"\n    lines = post_mortem(SOURCE, 'report([{\"day\": \"2026-02-01\", \"km\": 6.0}])')\n    assert lines == [\"KeyError: '2026-02'\", \"at line 5 in monthly: totals[month] += walk[\\\"km\\\"]\",\n                     \"  month = '2026-02'\", \"  totals = {}\", \"  walk = {'day': '2026-02-01', 'km': 6.0}\",\n                     \"  walks = [{'day': '2026-02-01', 'km': 6.0}]\"]\n\ndef test_fine():\n    \"no error\"\n    assert post_mortem(SOURCE, \"report([])\") == [\"no error: {}\"]\n    assert post_mortem(\"def f(x):\\n    return 1 / x\\n\", \"f(0)\")[:2] == [\"ZeroDivisionError: division by zero\", \"at line 2 in f: return 1 / x\"]")

## Warm-up

Every function uses the **same** `i`, looked up when it's called, after the loop has finished with `i == 2`. It's the most famous Python trap in interviews, and the reason today exists: the bug isn't where it looks.

## The method

1. **Read the error.** The last line of a traceback is the exception and its message; the frames above it go from the outermost call to the innermost, so the last `File` line is where it happened. Read the message literally: `KeyError: 'Ridge'` says which key.
2. **Reproduce** it with the smallest input you can (Week 95's minimising, or bisection over the input).
3. **Hypothesise** aloud: "I think `totals` is missing months with no walks".
4. **Test the hypothesis** with one experiment: a print, an assertion, a breakpoint (`breakpoint()` starts `pdb` on your own computer).
5. **Fix the cause**, not the symptom (a `try/except KeyError: pass` hides the bug).
6. **Add a test** that fails before the fix and passes after.

Say each step as you go; the interviewer is grading the method.

## Classic traps

- **Late-binding closures**: a function made in a loop sees the loop variable's final value. Bind it now with a default argument (`lambda x, i=i: x * i`).
- **Changing a list while looping over it**: removing items shifts the rest, so some are skipped. Build a new list instead.
- **Float equality**: `0.1 + 0.2 == 0.3` is `False`. Use `math.isclose`, or `Decimal` for money.
- **Numbers read as text**: input, CSV and JSON keys give strings, and `"9" > "10"` is `True`. Convert before comparing.
- **Shadowed built-ins**: `list = ...` or `sum = 0` replaces the built-in for the rest of the scope, and a later `list(...)` fails with `TypeError: 'list' object is not callable`.

## Practice

## Quiz

## Daily challenge

## A mock round

Ask a friend to break one of your old exercises (Week 88's walk log, or Week 91's incident report) in a way they don't tell you, and give you the failing test and twenty minutes. Say every step of the method aloud. Afterwards, write the test you'd add.

## Before you go

Tomorrow is the second behavioural session: harder questions about conflict, ambiguity and leading without authority, and the questions you'll ask them.